# step06_category_comparison — Historical Event Category Comparison

Compare the 31 article pages using one documented primary category per article.

Run cells from top to bottom. Shared helpers and file paths are in `analysis_common.py`. Outputs go to `analysis_results/` and `figures/`.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
ROOT = Path.cwd().resolve()
while not (ROOT / 'analysis' / 'analysis_common.py').exists() and ROOT.parent != ROOT: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'analysis'))
from analysis_common import *
df=classify_editors(load_revisions()).sort_values(['article_title','timestamp','revision_id'])
category=pd.DataFrame({'article_title':list(category_map()),'primary_category':list(category_map().values())})
article=[]
for title,g in df.groupby('article_title'):
    g=g.dropna(subset=['timestamp','article_size_bytes']); first,last=(g.iloc[0],g.iloc[-1]) if len(g) else (None,None)
    years=(last.timestamp-first.timestamp).total_seconds()/(365.2425*86400) if len(g)>1 else np.nan
    counts=g.username.value_counts()
    article.append({'article_title':title,'revisions':len(g),'unique_editors':g.username.nunique(),'avg_revisions_per_observed_year':len(g)/years if years and years>0 else np.nan,'first_size':first.article_size_bytes if first is not None else np.nan,'last_size':last.article_size_bytes if last is not None else np.nan,'growth_pct':100*(last.article_size_bytes-first.article_size_bytes)/first.article_size_bytes if first is not None and first.article_size_bytes else np.nan,'registered_pct':100*g.editor_type.eq('Registered (positive user ID)').mean(),'anonymous_pct':100*g.editor_type.eq('Anonymous (user ID 0)').mean(),'top10_share_pct':100*counts.head(10).sum()/len(g),'minor_pct':100*g.is_minor.mean()})
article=pd.DataFrame(article).merge(category,on='article_title',how='left',validate='one_to_one')
assert article.primary_category.notna().all(), 'Every analyzed article needs a primary category.'
save_table(category,'step06_article_category_mapping.csv'); save_table(article,'step06_article_category_metrics.csv'); display(category)


In [ ]:
category_summary=article.groupby('primary_category').agg(articles=('article_title','nunique'),total_revisions=('revisions','sum'),median_revisions_per_article=('revisions','median'),mean_revisions_per_article=('revisions','mean'),median_revisions_per_year=('avg_revisions_per_observed_year','median'),median_unique_editors=('unique_editors','median'),median_growth_pct=('growth_pct','median'),median_registered_pct=('registered_pct','median'),median_anonymous_pct=('anonymous_pct','median'),median_top10_share_pct=('top10_share_pct','median'),median_minor_pct=('minor_pct','median')).reset_index().sort_values('articles',ascending=False)
save_table(category_summary,'step06_category_summary.csv'); display(category_summary)
fig,ax=plt.subplots(figsize=(11,6)); category_summary.sort_values('median_revisions_per_article').plot.barh(x='primary_category',y='median_revisions_per_article',ax=ax,legend=False,color='#4f8d88'); ax.set(title='Median revisions per article by primary event category',xlabel='Median revision records',ylabel='Primary category'); finish_plot(fig,'step06_category_activity.png')
fig,ax=plt.subplots(figsize=(11,6)); article.boxplot(column='growth_pct',by='primary_category',ax=ax,rot=20); ax.set(title='Observed article growth by primary category',xlabel='Primary category',ylabel='Net size change (%)'); fig.suptitle(''); finish_plot(fig,'step06_category_growth.png')


## Interpretation

Primary categories are a documented, mutually exclusive convenience for comparison; several articles could reasonably fit elsewhere. Always show article counts and medians because category sizes differ and a few articles can dominate means. Category-level differences are descriptive and do not establish that category caused editing behavior.

In [ ]:
if article.minor_pct.nunique(dropna=False) <= 1:
    print('Minor-edit comparison is uninformative: the cleaned is_minor field is constant across all records.')
print('Category mapping counts:', category.groupby('primary_category').size().to_dict())
